# 0. Define Taxonomies & Data Model

Defines every classification taxonomy this project knows about and stands
up the canonical data model in Unity Catalog so the rest of the pipeline
(and the app) can read and write it uniformly. See `docs/DESIGN.md` for the
full model.

Outputs:
- `<name>_taxonomy` (one per schema, normalized to `code, label, level_path` + raw cols)
- `schema_registry` — one row per schema with metadata
- `predictions` — machine predictions, one row per `(order_id, schema_name, source)`, `source ∈ {llm, catboost}`
- `reviews` — append-only human review events (the audit log)
- `ground_truth` — the current accepted label per `(order_id, schema_name)`; CatBoost training set + LLM example source

Add a new taxonomy by writing a `ClassificationSchema` subclass in
`src/schemas/` and registering it. This notebook needs no code change.

In [ ]:
%pip install pyyaml openai openpyxl pydantic
%restart_python

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import pyspark.sql.functions as F
from src.utils import get_spark
from src.config import load_config
from src.schemas import list_schemas

spark = get_spark()
config = load_config()

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
spark.sql(f'USE {catalog}.{schema}')

## Load each taxonomy

In [ ]:
import re

def _sanitize(col: str) -> str:
    return re.sub(r'[^A-Za-z0-9_]+', '_', col).strip('_').lower()

registry_rows = []
for s in list_schemas():
    print(f'==> {s.name}: {s.display_name}')
    pdf = s.load_taxonomy()
    pdf['level_path'] = pdf['level_path'].apply(lambda xs: [str(x) for x in xs])
    for c in [c for c in pdf.columns if c not in ('code','label','level_path')]:
        pdf[c] = pdf[c].astype(str).fillna('')
    # Delta rejects spaces / special chars in column names.
    pdf = pdf.rename(columns={c: _sanitize(c) for c in pdf.columns})

    target = f'{catalog}.{schema}.{s.name}_taxonomy'
    (spark.createDataFrame(pdf)
        .withColumn('schema_name', F.lit(s.name))
        .write.format('delta').mode('overwrite').option('overwriteSchema','true')
        .saveAsTable(target))
    print(f'   {len(pdf):,} rows -> {target}')
    registry_rows.append({
        'name': s.name,
        'display_name': s.display_name,
        'description': s.spec.description,
        # Every schema now uses the single LLM + pgvector classifier
        # (plus optional CatBoost). No per-schema strategy anymore.
        'classify_method': 'llm_pgvector',
        'taxonomy_table': f'{s.name}_taxonomy',
        'level_columns': list(s.spec.level_columns),
        'code_column': s.spec.code_column,
        'label_column': s.spec.label_column,
        'description_columns': list(s.spec.description_columns),
        'leaf_count': int(len(pdf)),
    })

## Publish the schema registry

In [ ]:
registry_pdf = pd.DataFrame(registry_rows)
(spark.createDataFrame(registry_pdf)
    .write.format('delta').mode('overwrite').option('overwriteSchema','true')
    .saveAsTable(f'{catalog}.{schema}.schema_registry'))
registry_pdf

## Create the data-model tables

Three tables form the canonical model (see `docs/DESIGN.md` §4):

- **`predictions`** — machine predictions. One row per `(order_id, schema_name, source)`,
  `source ∈ {llm, catboost}`. Carries `status`/`reason` (so unclassified rows are
  self-explaining), `rationale` (the model's "thinking"), the retrieved `candidates`,
  `model_version`, and the MLflow `trace_id`.
- **`reviews`** — append-only log of human review *events* (`approved`/`corrected`/`rejected`).
  Never updated or deleted — this is the audit trail.
- **`ground_truth`** — the *current* accepted label per `(order_id, schema_name)`. Seeded in
  bulk by importing classified spend, grown by the review loop, and consumed as the CatBoost
  training set and the LLM's few-shot example source.

Created with `IF NOT EXISTS` so re-running the notebook never drops data.

In [ ]:
# predictions — machine predictions, keyed on (order_id, schema_name, source).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.predictions (
    order_id STRING,
    schema_name STRING,
    source STRING,                 -- 'llm' | 'catboost'
    code STRING,                   -- predicted leaf; NULL when unclassified
    label STRING,
    level_path ARRAY<STRING>,
    confidence DOUBLE,             -- normalized 0-1 across sources
    status STRING,                 -- 'classified' | 'unclassified'
    reason STRING,                 -- ok | low_confidence | no_candidates | parse_error | llm_error | out_of_taxonomy
    rationale STRING,              -- the model's short reasoning / "thinking"
    candidates ARRAY<STRUCT<code: STRING, label: STRING, score: DOUBLE>>,
    model_version STRING,          -- endpoint name / CatBoost run id
    trace_id STRING,               -- MLflow trace for this prediction
    classified_at TIMESTAMP
) USING DELTA
""")

# reviews — append-only human review events (the audit log).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.reviews (
    review_id STRING,
    order_id STRING,
    schema_name STRING,
    reviewed_source STRING,        -- which prediction the human looked at
    action STRING,                 -- 'approved' | 'corrected' | 'rejected'
    code STRING,                   -- the human's chosen label (approve/correct)
    label STRING,
    level_path ARRAY<STRING>,
    reviewer STRING,
    comment STRING,
    created_at TIMESTAMP
) USING DELTA
""")

# ground_truth — current accepted label per (order_id, schema_name).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.ground_truth (
    order_id STRING,
    schema_name STRING,
    code STRING,
    label STRING,
    level_path ARRAY<STRING>,
    origin STRING,                 -- 'imported' | 'human_review'
    review_id STRING,              -- FK -> reviews when origin='human_review'
    labeled_by STRING,
    labeled_at TIMESTAMP
) USING DELTA
""")

for t in ('predictions', 'reviews', 'ground_truth'):
    print(f'== {t} ==')
    spark.sql(f'DESCRIBE {catalog}.{schema}.{t}').display()